In [1]:
import os
import numpy as np
import pandas as pd
import random as rnd

import nltk
from nltk.corpus import stopwords #not used here!
from nltk.tokenize import word_tokenize 

#nltk.download('punkt')
#nltk.download('stopwords')

In [2]:
data = pd.read_csv("questions_answers.csv", encoding='cp1252') ### You will need to put in your own question & answer data here ###
N=len(data)
print('Number of question pairs: ', N)
data.head(351)

Number of question pairs:  100


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,Failed to build object mandatory component: Co...,Trader to remark vol surface for the underlying,1
1,1,3,4,Instrument is forward starting but start date ...,TA to rebook without forward start or forward ...,1
2,2,5,6,Failed to build object mandatory component: Co...,TA to check if either missing a credit underly...,1
3,3,7,8,Invalid or missing cdsParSpreads: Underlying 9...,TA to check if the ISDA41/Liquid curve/CDS spr...,1
4,4,9,10,Failed to build object mandatory component: Pa...,"TA to fix - FX timing rule set to ""EXCHANGE DE...",1
...,...,...,...,...,...,...
95,95,191,192,EdgIRSmileCalibNew: invalid combination of exo...,This is due to some missing IR Q-smile data.,1
96,96,193,194,Failed to price:RainbowKOMC: Rebate pay at hit...,"The rebate pay at hit flag indicates whether, ...",1
97,97,195,196,ScalarShift::calculateCrossDerivative: Failed ...,This is usually related to the Monte Carlo imp...,1
98,98,197,198,IlliquidStock::getProcessedVol: illiquid stock...,This means the trader needs to set up the vola...,1


In [3]:
print(data['question1'][5])  #  Example of question duplicates (6th one in data)
print(data['question2'][5])

X = input("Enter first question: ")
Y = np.array(data['question1']) # assign whole question1 array to Y
Z = np.array(data['question2']) # assign whole question2 array to Z (which are answers to question1)

#print(Y[1])

The barrier is very close to the lowest spot price, which can cause unstable numerical solutions. Please update the model parameters
TA to check, for an exchangeable bonds if the issuer is different from the underlying then the field risk growth would be N.
Enter first question: AssetHIstoryContainer::findHistoryForSource: No asset history exists for this asset. Failed FXAsset::pastValue: Failed to retrieve sample for asset equity-481795 on 19-Sep-2019


In [4]:
#define cosine similarity function first
def cosine_similarity(A, B):
    '''
    Input:
        A: a numpy array which corresponds to a word vector
        B: A numpy array which corresponds to a word vector
    Output:
        cos: numerical number representing the cosine similarity between A and B.
    '''

    ### START CODE HERE (REPLACE INSTANCES OF 'None' with your code) ###
    
    dot = np.dot(A,B)
    norma = np.linalg.norm(A)
    normb = np.linalg.norm(B)
    cos = dot/(norma*normb)

    ### END CODE HERE ###
    return cos

In [7]:
# Program to measure the similarity between two sentences using cosine similarity. 

#Question1 contains a list of sentences (commonly seen pricing errors) which are used to match for 
#user's input sentence (X) and suggest the solution from Question2

# tokenization input sentence and question1 array sentences 
X_list = word_tokenize(X)  

#create arrays
Y_list = np.empty_like(Y)

for idx in range(len(Y)):
    Y_list[idx] = nltk.word_tokenize(Y[idx])

#print(X_list)
#print(Y_list[99])

# sw contains the list of stopwords 
#sw = stopwords.words('english')
#print(sw)
    
# remove stop words from the string 
#X_set = {w for w in X_list if not w in sw}
#Y_set = {w for w in Y_list if not w in sw}
#print(X_set)
#print(Y_set)

#convert list into strings again
X_set = {w for w in X_list}
#print(X_set)

cosine_new=[None]*len(Y)
id = 0
cosine_initial = 0 

for idx in range(len(Y)):
    a= Y_list[idx] 
    Y_set = {w for w in a}
    #print(Y_set)     
    
    l1 =[];l2 =[]
    rvector= X_set.union(Y_set)  
    for w in rvector: 
        if w in X_set: l1.append(1) # create a vector 
        else: l1.append(0) 
        if w in Y_set: l2.append(1) 
        else: l2.append(0)
    
    cosine_new = cosine_similarity(l1, l2)
    
    if cosine_new > cosine_initial:
        cosine_initial = cosine_new
        id = idx
        print("similarity: ", id, cosine_initial)

if cosine_initial >= 0.5:
    print("The answer to your question is:    ", Z[id])
else:
    print("Sorry, I don't have an answer for you right now, please consult L1 team.")
        
    

similarity:  0 0.1720618004029213
similarity:  2 0.18394180184548972
similarity:  4 0.2075143391598224
similarity:  12 0.22941573387056174
similarity:  19 0.3077935056255462
similarity:  67 0.8947368421052629
The answer to your question is:     Duplicate entries in CAH for XFX should be deleted by Operate, then the instrument can be repriced.
